In [1]:
import os
os.chdir("..")
print(os.getcwd())

c:\Users\NASA2004\OneDrive\Documents\transformer-from-scratch


In [2]:
# importing dataset from hugging face
from datasets import load_dataset

ds = load_dataset("bentrevett/multi30k")

In [ ]:
# checking the dataset split
print(ds)

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})


In [ ]:
# saving the dataset as train/val/test
import os

def write_split(dataset_split, split_name, out_dir="data/raw"):
    os.makedirs(out_dir, exist_ok=True)
    en_path = os.path.join(out_dir, f"{split_name}.en")
    de_path = os.path.join(out_dir, f"{split_name}.de")
    with open(en_path, "w", encoding="utf-8") as f_en, \
         open(de_path, "w", encoding="utf-8") as f_de:
        for example in dataset_split:
            f_en.write(example["en"].strip() + "\n")
            f_de.write(example["de"].strip() + "\n")
    print(f"{split_name}: wrote {len(dataset_split)} lines to {en_path} and {de_path}")

write_split(ds["train"], "train")
write_split(ds["validation"], "val")
write_split(ds["test"], "test")

train: wrote 29000 lines to data/raw\train.en and data/raw\train.de
val: wrote 1014 lines to data/raw\val.en and data/raw\val.de
test: wrote 1000 lines to data/raw\test.en and data/raw\test.de


In [5]:
with open("data/raw/train.en") as f:
    lines = f.readlines()
print(len(lines), lines[0])

29000 Two young, White males are outside near many bushes.



In [ ]:
# building a byte-pair encoder (bpe) tokenizer
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=8000,
    special_tokens=["<pad>", "<bos>", "<eos>", "<unk>"]
)

tokenizer.train(files=["data/raw/train.en", "data/raw/train.de"], trainer=trainer)

os.makedirs("data/tokenizer", exist_ok=True)
tokenizer.save("data/tokenizer/multi30k_bpe.json")

In [11]:
# handling special tokens
from tokenizers.processors import TemplateProcessing

bos_id = tokenizer.token_to_id("<bos>")
eos_id = tokenizer.token_to_id("<eos>")

tokenizer.post_processor = TemplateProcessing(
    single="<bos> $A <eos>",
    special_tokens=[
        ("<bos>", bos_id),
        ("<eos>", eos_id),
    ],
)

# re-checking
encoded = tokenizer.encode("Two young, White males are outside near many bushes.")
print(encoded.tokens)
print(encoded.ids)

# re-saving
tokenizer.save("data/tokenizer/multi30k_bpe.json")

['<bos>', 'Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.', '<eos>']
[1, 217, 273, 13, 5008, 3650, 192, 479, 581, 1788, 5743, 15, 2]


In [12]:
# verifying english tokens
tokenizer = Tokenizer.from_file("data/tokenizer/multi30k_bpe.json")

encoded = tokenizer.encode("Two young, White males are outside near many bushes.")
print(encoded.tokens)
print(encoded.ids)

['<bos>', 'Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.', '<eos>']
[1, 217, 273, 13, 5008, 3650, 192, 479, 581, 1788, 5743, 15, 2]


In [13]:
# verifying german tokens
tokenizer = Tokenizer.from_file("data/tokenizer/multi30k_bpe.json")

encoded_de = tokenizer.encode("Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.")
print(encoded_de.tokens)
print(encoded_de.ids)

['<bos>', 'Zwei', 'junge', 'weiße', 'Männer', 'sind', 'im', 'Freien', 'in', 'der', 'Nähe', 'viel', 'er', 'Bü', 'sche', '.', '<eos>']
[1, 214, 544, 1392, 306, 708, 168, 743, 103, 145, 839, 1662, 105, 813, 2370, 15, 2]


In [ ]:
# wrapping the tokenizer
import torch
from torch.utils.data import Dataset

class Multi30kDataset(Dataset):
    def __init__(self, en_path, de_path, tokenizer):
        self.tokenizer = tokenizer

        with open(en_path, encoding="utf-8") as f:
            en_sentences = [line.strip() for line in f]
        with open(de_path, encoding="utf-8") as f:
            de_sentences = [line.strip() for line in f]

        assert len(en_sentences) == len(de_sentences), "Mismatched EN/DE line counts"

        self.en_ids = [tokenizer.encode(s).ids for s in en_sentences]
        self.de_ids = [tokenizer.encode(s).ids for s in de_sentences]

    def __len__(self):
        return len(self.en_ids)

    def __getitem__(self, idx):
        src = torch.tensor(self.en_ids[idx], dtype=torch.long)
        tgt = torch.tensor(self.de_ids[idx], dtype=torch.long)
        return src, tgt

In [ ]:
# verifying
train_dataset = Multi30kDataset("data/raw/train.en", "data/raw/train.de", tokenizer)
print(len(train_dataset))
print(train_dataset[0])

29000
(tensor([   1,  217,  273,   13, 5008, 3650,  192,  479,  581, 1788, 5743,   15,
           2]), tensor([   1,  214,  544, 1392,  306,  708,  168,  743,  103,  145,  839, 1662,
         105,  813, 2370,   15,    2]))


In [16]:
# padding
from torch.nn.utils.rnn import pad_sequence

PAD_ID = tokenizer.token_to_id("<pad>")

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=PAD_ID)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_ID)

    tgt_input = tgt_padded[:, :-1]   # decoder input
    tgt_label = tgt_padded[:, 1:]    # decoder label

    return src_padded, tgt_input, tgt_label

In [17]:
# verifying
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

src, tgt_input, tgt_label = next(iter(train_loader))
print(src.shape, tgt_input.shape, tgt_label.shape)

torch.Size([32, 37]) torch.Size([32, 32]) torch.Size([32, 32])


In [18]:
# padding masks
def create_masks(src, tgt_input, pad_id):
    # src padding mask
    src_padding_mask = (src != pad_id).unsqueeze(1).unsqueeze(2)

    # tgt padding mask
    tgt_padding_mask = (tgt_input != pad_id).unsqueeze(1).unsqueeze(3)

    tgt_len = tgt_input.size(1)
    # causal mask
    causal_mask = torch.tril(torch.ones(tgt_len, tgt_len, dtype=torch.bool)).unsqueeze(0).unsqueeze(0)

    # combining
    tgt_mask = tgt_padding_mask & causal_mask

    return src_padding_mask, tgt_mask

In [19]:
# verifying
src_mask, tgt_mask = create_masks(src, tgt_input, PAD_ID)
print(src_mask.shape, tgt_mask.shape)

torch.Size([32, 1, 1, 37]) torch.Size([32, 1, 32, 32])
